# Week 60, Day 4: Hierarchical and spectral clustering

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w060/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Average linkage from scratch

Write agglomerate(X), which keeps a list of clusters (lists of point indices), finds the pair with the smallest average pairwise distance, merges it, records the distance, and repeats until one cluster remains. Check the merge heights against scipy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

def agglomerate(X):
    D = np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1))
    clusters = [[i] for i in range(len(X))]
    heights = []
    while len(clusters) > 1:
        best = (np.inf, 0, 1)
        for a in range(len(clusters)):
            for b in range(a + 1, len(clusters)):
                d = ...                                   # average distance between clusters a and b
                if d < best[0]:
                    best = (d, a, b)
        d, a, b = best
        heights.append(d)
        clusters[a] = clusters[a] + clusters[b]
        del clusters[b]
    return heights

g1 = np.random.default_rng(604)
Xs = np.vstack([g1.normal([0, 0], 0.5, (10, 2)), g1.normal([3, 0], 0.5, (10, 2)), g1.normal([1.5, 3], 0.5, (10, 2))])
heights = agglomerate(Xs)
print("last five merge heights:", np.round(heights[-5:], 2))
dendrogram(linkage(Xs, method="average"), no_labels=True); plt.ylabel("merge distance"); plt.show()

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage
def test_heights():
    "the merge heights match scipy's average linkage"
    ref = np.sort(linkage(Xs, method="average")[:, 2])
    assert np.allclose(np.sort(heights), ref)

test_heights()
print("All checks passed")

### Exercise: Linkage on rings

Make two rings (radii 1 and 3, 150 points each, noise 0.1). Cut single and complete linkage trees into two clusters and score them against the rings. Then add 15 points scattered between the rings and repeat single linkage.

In [ ]:
def rings(n, g, noise=0.1):
    a = g.uniform(0, 2 * np.pi, 2 * n)
    r = np.r_[np.full(n, 1.0), np.full(n, 3.0)]
    return np.column_stack([r * np.cos(a), r * np.sin(a)]) + noise * g.normal(size=(2 * n, 2)), np.r_[np.zeros(n, int), np.ones(n, int)]

def acc2(lab, truth):
    a = np.mean((lab - lab.min()) == truth)
    return max(a, 1 - a)

g2 = np.random.default_rng(6041)
Xr, yr = rings(150, g2)
acc_single = acc2(fcluster(linkage(Xr, "single"), 2, criterion="maxclust"), yr)
acc_complete = ...
ang = g2.uniform(0, 2 * np.pi, 15); rad = g2.uniform(1.3, 2.7, 15)
bridge = np.column_stack([rad * np.cos(ang), rad * np.sin(ang)])
lab_b = fcluster(linkage(np.vstack([Xr, bridge]), "single"), 2, criterion="maxclust")[:300]
acc_single_bridge = acc2(lab_b, yr)
print(f"single {acc_single:.3f}, complete {acc_complete:.3f}; single with 15 bridging points {acc_single_bridge:.3f}")

In [ ]:
import numpy as np
def test_single():
    "single linkage separates clean rings perfectly"
    assert acc_single == 1.0
def test_others():
    "complete linkage does not"
    assert acc_complete < 0.8
def test_chain():
    "with outlying points between the rings, single linkage chains them together"
    assert acc_single_bridge < 0.8

test_single()
test_others()
test_chain()
print("All checks passed")

### Exercise: Spectral clustering of rings

Build the Gaussian affinity matrix (sigma 0.3, zero diagonal) for the rings with bridging points, form L_sym, take the eigenvectors of its two smallest eigenvalues, normalise the rows, and cluster them with k-means (k = 2).

In [ ]:
from scipy.cluster.vq import kmeans2
Xb = np.vstack([Xr, bridge])
sigma = 0.3
D2 = ((Xb[:, None] - Xb[None]) ** 2).sum(-1)
W = np.exp(-D2 / (2 * sigma ** 2)); np.fill_diagonal(W, 0)
d = W.sum(1)
Lsym = ...
vals, vecs = np.linalg.eigh(Lsym)
U = vecs[:, :2]
U = ...
_, lab_spec = kmeans2(U, 2, minit="++", seed=1)
acc_spec = acc2(lab_spec[:300], yr)
print("smallest eigenvalues:", np.round(vals[:4], 4))
print(f"spectral clustering agreement with the rings: {acc_spec:.3f}")

In [ ]:
import numpy as np
def test_lap():
    "the normalised Laplacian is symmetric with smallest eigenvalue near 0"
    assert np.allclose(Lsym, Lsym.T) and abs(np.linalg.eigvalsh(Lsym)[0]) < 1e-8
def test_spectral():
    "spectral clustering separates the rings even with the bridging points"
    assert acc_spec > 0.95

test_lap()
test_spectral()
print("All checks passed")